In [5]:
!unzip -q /content/restaurant-insight-absa.zip -d /content/
%cd /content/restaurant-insight-absa


replace /content/restaurant-insight-absa/HANDOFF_TO_CLAUDE_CODE.md? [y]es, [n]o, [A]ll, [N]one, [r]ename: /content/restaurant-insight-absa


In [6]:
!pip install -q -r requirements.txt !python -m spacy download en_core_web_md import nltk nltk.download('punkt'); nltk.download('punkt_tab'); nltk.download('averaged_perceptron_tagger'); nltk.download('averaged_perceptron_tagger_eng'); nltk.download('stopwords')

/bin/bash: -c: line 1: syntax error near unexpected token `('
/bin/bash: -c: line 1: `pip install -q -r requirements.txt !python -m spacy download en_core_web_md import nltk nltk.download('punkt'); nltk.download('punkt_tab'); nltk.download('averaged_perceptron_tagger'); nltk.download('averaged_perceptron_tagger_eng'); nltk.download('stopwords')'


In [8]:
!python -m spacy download en_core_web_md

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 33.5/33.5 MB 40.0 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_md')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [9]:
import re
path = "src/rating_classifier.py"
with open(path) as f:
    content = f.read()
content = content.replace(
    '''    report = classification_report(
        y_test, y_pred, target_names=[LABEL_NAMES[i] for i in sorted(LABEL_NAMES)], output_dict=True
    )
    cm = confusion_matrix(y_test, y_pred)''',
    '''    all_labels = sorted(LABEL_NAMES)
    report = classification_report(
        y_test,
        y_pred,
        labels=all_labels,
        target_names=[LABEL_NAMES[i] for i in all_labels],
        output_dict=True,
        zero_division=0,
    )
    cm = confusion_matrix(y_test, y_pred, labels=all_labels)'''
)
with open(path, "w") as f:
    f.write(content)
print("patched")

patched


In [10]:
%%writefile src/transformer_rating_classifier.py
"""
DeBERTa-v3 fine-tuning for review-level rating sentiment (Negative/Neutral/
Positive), built to replace the original prototype's hand-rolled training
loop — which the audit found to contain a syntax error in its own metric
print statement, an unfilled placeholder model path, and suspiciously
perfectly-linear per-epoch loss values inconsistent with real training
(see AUDIT.md section C).

This version uses Hugging Face's `Trainer`, which:
  - logs real per-step/per-epoch loss automatically (no hand-typed numbers),
  - computes and logs real accuracy/F1 every epoch via `compute_metrics`,
  - saves a `trainer_state.json` log history as hard evidence of the run,
  - saves the fine-tuned model + tokenizer together with `save_pretrained`
    (fixing the original's unfilled `<<INSERT MODEL NAME HERE>>` path bug).

Uses the SAME data-cleaning and train/test split logic as
`rating_classifier.py` (same seed, same stratification) so the two are
directly, fairly comparable on identical data.

Requires `transformers`, `torch`, and ideally a GPU. Import is lazy so
this module doesn't fail to import in environments without a GPU;
actually calling `train_transformer()` does need those packages.
"""

from __future__ import annotations

import json
from pathlib import Path

import numpy as np
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
)
from sklearn.model_selection import train_test_split

from .rating_classifier import LABEL_NAMES, RANDOM_SEED, load_and_clean

DEFAULT_MODEL_NAME = "microsoft/deberta-v3-base"


class _RatingDataset:
    """Minimal torch Dataset wrapping tokenized text + integer labels."""

    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        import torch

        item = {k: torch.tensor(v[idx]) for k, v in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item


def _compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "macro_f1": f1_score(labels, preds, average="macro"),
        "weighted_f1": f1_score(labels, preds, average="weighted"),
    }


def train_transformer(
    csv_path: str,
    model_name: str = DEFAULT_MODEL_NAME,
    output_dir: str = "models/deberta_rating_sentiment",
    num_epochs: int = 3,
    batch_size: int = 16,
    max_length: int = 128,
    val_size: float = 0.1,
    test_size: float = 0.2,
) -> dict:
    """
    Fine-tune `model_name` (default DeBERTa-v3-base) for 3-class rating
    sentiment. Returns a dict with the same shape as
    `rating_classifier.train()`'s return value (minus `pipeline`, plus
    `training_log`) so the two can be compared directly.
    """
    import torch
    from transformers import (
        AutoModelForSequenceClassification,
        AutoTokenizer,
        Trainer,
        TrainingArguments,
    )

    df = load_and_clean(csv_path)

    # Same split logic/seed as rating_classifier.train() for a fair,
    # apples-to-apples comparison — train/val carved out of the same
    # train partition the classical baseline uses, test partition identical.
    X_trainval, X_test, y_trainval, y_test = train_test_split(
        df["Review"], df["label"], test_size=test_size, random_state=RANDOM_SEED, stratify=df["label"]
    )
    X_train, X_val, y_train, y_val = train_test_split(
        X_trainval, y_trainval, test_size=val_size, random_state=RANDOM_SEED, stratify=y_trainval
    )

    overlap = (set(X_train) & set(X_test)) | (set(X_val) & set(X_test)) | (set(X_train) & set(X_val))
    assert not overlap, f"Data leakage across splits: {len(overlap)} overlapping reviews"

    tokenizer = AutoTokenizer.from_pretrained(model_name)

    def tokenize(texts):
        return tokenizer(
            list(texts), truncation=True, padding="max_length", max_length=max_length
        )

    train_ds = _RatingDataset(tokenize(X_train), y_train.tolist())
    val_ds = _RatingDataset(tokenize(X_val), y_val.tolist())
    test_ds = _RatingDataset(tokenize(X_test), y_test.tolist())

    model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=3)

    args = TrainingArguments(
        output_dir=f"{output_dir}_checkpoints",
        num_train_epochs=num_epochs,
        per_device_train_batch_size=batch_size,
        per_device_eval_batch_size=batch_size,
        eval_strategy="epoch",
        save_strategy="epoch",
        logging_strategy="steps",
        logging_steps=25,
        load_best_model_at_end=True,
        metric_for_best_model="macro_f1",
        seed=RANDOM_SEED,
        report_to=[],  # no wandb/etc — keep this self-contained
        # DeBERTa-v3's disentangled attention is numerically unstable
        # under fp16 (a widely reported HF issue: relative-position score
        # computations overflow, weights go NaN within the first few
        # steps, loss collapses to 0.0/NaN and predictions freeze on one
        # class). Force pure fp32 to avoid this, and use a lower,
        #

Overwriting src/transformer_rating_classifier.py


In [11]:
!pip install -q accelerate

In [22]:
from src import transformer_rating_classifier as trc
result = trc.train_transformer("data/Restaurant reviews.csv")
print("macro_f1:", result["macro_f1"])
print("weighted_f1:", result["weighted_f1"])
print("accuracy:", result["classification_report"]["accuracy"])

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

[transformers] DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     | 
----------------------------------------+------------+-
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
pooler.dense.bias                       | MISSING    | 
classifier.bias                         | MISSING    | 
pooler.dense.weight                     | MISSING    | 
classifier

Epoch,Training Loss,Validation Loss,Accuracy,Macro F1,Weighted F1
1,0.311505,0.349631,0.863818,0.741478,0.853608
2,0.277773,0.370023,0.869159,0.761011,0.862091
3,0.299398,0.381708,0.865154,0.758711,0.860204


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

macro_f1: 0.7730208013974326
weighted_f1: 0.8711012236496275
accuracy: 0.8798718633208756


In [12]:
file_path = "/content/restaurant-insight-absa/src/transformer_rating_classifier.py"

with open(file_path, "r") as f:
    code = f.read()

# Remove warmup_ratio from TrainingArguments
import re
code = re.sub(r'^\s*warmup_ratio\s*=\s*[^,\n]+,\s*\n', '', code, flags=re.MULTILINE)

with open(file_path, "w") as f:
    f.write(code)

print("✅ warmup_ratio removed successfully.")

✅ warmup_ratio removed successfully.


In [13]:
file_path = "/content/restaurant-insight-absa/src/transformer_rating_classifier.py"

with open(file_path, "r") as f:
    code = f.read()

if "warmup_ratio" in code:
    print("❌ warmup_ratio is still present")
else:
    print("✅ warmup_ratio is no longer in the file")

✅ warmup_ratio is no longer in the file


In [15]:
import importlib
from src import transformer_rating_classifier as trc

importlib.reload(trc)

print("✅ Module reloaded")

SyntaxError: '(' was never closed (transformer_rating_classifier.py, line 124)

In [16]:
!grep -n "def train_transformer\|return result" src/transformer_rating_classifier.py

72:def train_transformer(


In [17]:
!grep -n "def train_transformer" src/transformer_rating_classifier.py

72:def train_transformer(


In [21]:
!python -c "import transformers; print(transformers.__version__)"

5.15.1


In [22]:
!python -c "import ast; ast.parse(open('src/transformer_rating_classifier.py').read()); print('syntax OK')"

Traceback (most recent call last):
  File "<string>", line 1, in <module>
    import ast; ast.parse(open('src/transformer_rating_classifier.py').read()); print('syntax OK')
                ~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.13/ast.py", line 50, in parse
    return compile(source, filename, mode, flags,
                   _feature_version=feature_version, optimize=optimize)
  File "<unknown>", line 124
    args = TrainingArguments(
                            ^
SyntaxError: '(' was never closed


In [23]:
import re

path = "src/transformer_rating_classifier.py"
with open(path) as f:
    content = f.read()

content = content.replace(
    '        learning_rate=1e-5,\n        warmup_ratio=0.1,\n        max_grad_norm=1.0,',
    '        learning_rate=1e-5,\n        max_grad_norm=1.0,'
)

with open(path, "w") as f:
    f.write(content)

print("patched")

patched


In [24]:
!wc -l src/transformer_rating_classifier.py
!tail -15 src/transformer_rating_classifier.py

142 src/transformer_rating_classifier.py
        per_device_eval_batch_size=batch_size,
        eval_strategy="epoch",
        save_strategy="epoch",
        logging_strategy="steps",
        logging_steps=25,
        load_best_model_at_end=True,
        metric_for_best_model="macro_f1",
        seed=RANDOM_SEED,
        report_to=[],  # no wandb/etc — keep this self-contained
        # DeBERTa-v3's disentangled attention is numerically unstable
        # under fp16 (a widely reported HF issue: relative-position score
        # computations overflow, weights go NaN within the first few
        # steps, loss collapses to 0.0/NaN and predictions freeze on one
        # class). Force pure fp32 to avoid this, and use a lower,
        #


In [25]:
%%writefile src/transformer_rating_classifier.py
"""
DeBERTa-v3 fine-tuning for review-level rating sentiment (Negative/Neutral/
Positive), built to replace the original prototype's hand-rolled training
loop — which the audit found to contain a syntax error in its own metric
print statement, an unfilled placeholder model path, and suspiciously
perfectly-linear per-epoch loss values inconsistent with real training
(see AUDIT.md section C).

This version uses Hugging Face's `Trainer`, which:
  - logs real per-step/per-epoch loss automatically (no hand-typed numbers),
  - computes and logs real accuracy/F1 every epoch via `compute_metrics`,
  - saves a `trainer_state.json` log history as hard evidence of the run,
  - saves the fine-tuned model + tokenizer together with `save_pretrained`
    (fixing the original's unfilled `<<INSERT MODEL NAME HERE>>` path bug).

Uses the SAME data-cleaning and train/test split logic as
`rating_classifier.py` (same seed, same stratification) so the two are
directly, fairly comparable on identical data.

Requires `transformers`, `torch`, and ideally a GPU. Import is lazy so
this module doesn't fail to import in environments without a GPU;
actually calling `train_transformer()` does need those packages.
"""

from __future__ import annotations

import json
from pathlib import Path

import numpy as np
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
)
from sklearn.model_selection import train_test_split

from .rating_classifier import LABEL_NAMES, RANDOM_SEED, load_and_clean

DEFAULT_MODEL_NAME = "microsoft/deberta-v3-base"


class _RatingDataset:
    """Minimal torch Dataset wrapping tokenized text + integer labels."""

    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        import torch

        item = {k: torch.tensor(v[idx]) for k, v in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item


def _compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "macro_f1": f1_score(labels, preds, average="macro"),
        "weighted_f1": f1_score(labels, preds, average="weighted"),
    }


def train_transformer(
    csv_path: str,
    model_name: str = DEFAULT_MODEL_NAME,
    output_dir: str = "models/deberta_rating_sentiment",
    num_epochs: int = 3,
    batch_size: int = 16,
    max_length: int = 128,
    val_size: float = 0.1,
    test_size: float = 0.2,
) -> dict:
    """
    Fine-tune `model_name` (default DeBERTa-v3-base) for 3-class rating
    sentiment. Returns a dict with the same shape as
    `rating_classifier.train()`'s return value (minus `pipeline`, plus
    `training_log`) so the two can be compared directly.
    """
    import torch
    from transformers import (
        AutoModelForSequenceClassification,
        AutoTokenizer,
        Trainer,
        TrainingArguments,
    )

Overwriting src/transformer_rating_classifier.py


In [26]:
%%writefile -a src/transformer_rating_classifier.py

    df = load_and_clean(csv_path)

    # Same split logic/seed as rating_classifier.train() for a fair,
    # apples-to-apples comparison — train/val carved out of the same
    # train partition the classical baseline uses, test partition identical.
    X_trainval, X_test, y_trainval, y_test = train_test_split(
        df["Review"], df["label"], test_size=test_size, random_state=RANDOM_SEED, stratify=df["label"]
    )
    X_train, X_val, y_train, y_val = train_test_split(
        X_trainval, y_trainval, test_size=val_size, random_state=RANDOM_SEED, stratify=y_trainval
    )

    overlap = (set(X_train) & set(X_test)) | (set(X_val) & set(X_test)) | (set(X_train) & set(X_val))
    assert not overlap, f"Data leakage across splits: {len(overlap)} overlapping reviews"

    tokenizer = AutoTokenizer.from_pretrained(model_name)

    def tokenize(texts):
        return tokenizer(
            list(texts), truncation=True, padding="max_length", max_length=max_length
        )

    train_ds = _RatingDataset(tokenize(X_train), y_train.tolist())
    val_ds = _RatingDataset(tokenize(X_val), y_val.tolist())
    test_ds = _RatingDataset(tokenize(X_test), y_test.tolist())

    model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=3)

    args = TrainingArguments(
        output_dir=f"{output_dir}_checkpoints",
        num_train_epochs=num_epochs,
        per_device_train_batch_size=batch_size,
        per_device_eval_batch_size=batch_size,
        eval_strategy="epoch",
        save_strategy="epoch",
        logging_strategy="steps",
        logging_steps=25,
        load_best_model_at_end=True,
        metric_for_best_model="macro_f1",
        seed=RANDOM_SEED,
        report_to=[],  # no wandb/etc — keep this self-contained
        # DeBERTa-v3's disentangled attention is numerically unstable
        # under fp16 (a widely reported HF issue: relative-position score
        # computations overflow, weights go NaN within the first few
        # steps, loss collapses to 0.0/NaN and predictions freeze on one
        # class). Force pure fp32 to avoid this, and use a lower,
        # DeBERTa-appropriate learning rate with warmup + gradient
        # clipping for stability.
        fp16=False,
        bf16=False,
        learning_rate=1e-5,
        warmup_ratio=0.1,
        max_grad_norm=1.0,
    )

    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=train_ds,
        eval_dataset=val_ds,
        compute_metrics=_compute_metrics,
    )

    trainer.train()

    # Real log history — actual per-step loss and per-epoch eval metrics,
    # straight from Trainer's internal state. This is the evidence file;
    # compare it visually to AUDIT.md section C's description of the
    # original notebook's suspiciously linear fabricated losses.
    training_log = trainer.state.log_history

    test_output = trainer.predict(test_ds)
    y_pred = np.argmax(test_output.predictions, axis=-1)

    all_labels = sorted(LABEL_NAMES)
    report = classification_report(
        y_test,
        y_pred,
        labels=all_labels,
        target_names=[LABEL_NAMES[i] for i in all_labels],
        output_dict=True,
        zero_division=0,
    )
    cm = confusion_matrix(y_test, y_pred, labels=all_labels)
    macro_f1 = f1_score(y_test, y_pred, average="macro")
    weighted_f1 = f1_score(y_test, y_pred, average="weighted")

    Path(output_dir).mkdir(parents=True, exist_ok=True)
    trainer.save_model(output_dir)
    tokenizer.save_pretrained(output_dir)
    with open(f"{output_dir}/training_log.json", "w") as f:
        json.dump(training_log, f, indent=2)

    result = {
        "model_name": model_name,
        "classification_report": report,
        "confusion_matrix": cm.tolist(),
        "macro_f1": macro_f1,
        "weighted_f1": weighted_f1,
        "n_train": len(X_train),
        "n_val": len(X_val),
        "n_test": len(X_test),
        "training_log": training_log,
    }
    with open(f"{output_dir}/metrics.json", "w") as f:
        json.dump({k: v for k, v in result.items() if k != "training_log"}, f, indent=2)

    return result

Appending to src/transformer_rating_classifier.py


In [27]:
!wc -l src/transformer_rating_classifier.py   # should say 204
!python -c "import ast; ast.parse(open('src/transformer_rating_classifier.py').read()); print('syntax OK')"

203 src/transformer_rating_classifier.py
syntax OK


In [28]:
path = "src/transformer_rating_classifier.py"
with open(path) as f:
    content = f.read()

content = content.replace(
    '    model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=3)',
    '''    model = AutoModelForSequenceClassification.from_pretrained(
        model_name,
        num_labels=3,
        torch_dtype=torch.float32,
        attn_implementation="eager",
    )'''
)

content = content.replace(
    '        learning_rate=1e-5,\n        max_grad_norm=1.0,',
    '        learning_rate=1e-5,\n        warmup_steps=100,\n        max_grad_norm=1.0,'
)

with open(path, "w") as f:
    f.write(content)

print("patched")

patched


In [29]:
import os
print("cwd:", os.getcwd())
print()
print("/content contents:", os.listdir("/content"))

cwd: /content/restaurant-insight-absa

/content contents: ['.config', 'restaurant-insight-absa.zip', 'restaurant-insight-absa', 'sample_data']


In [30]:
%cd /content/restaurant-insight-absa

/content/restaurant-insight-absa


In [31]:
!python -c "import ast; ast.parse(open('src/transformer_rating_classifier.py').read()); print('syntax OK')"
!grep -n "attn_implementation\|torch_dtype\|warmup_steps" src/transformer_rating_classifier.py

syntax OK
125:        torch_dtype=torch.float32,
126:        attn_implementation="eager",


In [35]:
%cd /content

/content


In [36]:
!unzip -q -o /content/restaurant-insight-absa.zip -d /content/
%cd /content/restaurant-insight-absa
!grep -n "warmup_steps\|warmup_ratio" src/transformer_rating_classifier.py

/content/restaurant-insight-absa
152:        warmup_steps=100,


In [37]:
import importlib
from src import transformer_rating_classifier as trc
importlib.reload(trc)

result = trc.train_transformer("data/Restaurant reviews.csv")
print("macro_f1:", result["macro_f1"])
print("weighted_f1:", result["weighted_f1"])
print("accuracy:", result["classification_report"]["accuracy"])

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

[transformers] DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     | 
----------------------------------------+------------+-
mask_predictions.dense.weight           | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
classifier.bias                         | MISSING    | 
classifier.weight                       | MISSING    | 
pooler.dense.weight                     | MISSING    | 
pooler.den

Epoch,Training Loss,Validation Loss,Accuracy,Macro F1,Weighted F1
1,0.319597,0.341928,0.866489,0.762558,0.861092
2,0.272554,0.371515,0.866489,0.749982,0.856946
3,0.283585,0.377355,0.867824,0.758058,0.861455


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

macro_f1: 0.7737855445450014
weighted_f1: 0.8708235776808274
accuracy: 0.8782701548318206
